# Forus DS Take-Home — Part 1: Go-to-Market Specialty Analysis

**What this notebook does**
1. Builds (or reuses) the analytics layer in BigQuery from the read-only source dataset.
2. Validates it against the source.
3. Runs the specialty economics from one editable **assumptions** cell.
4. Produces the specialty scorecard, a capture curve, sensitivity checks and a starter targeting list.

**To run it yourself:** set `BILLING_PROJECT` and `TARGET` in the next cell to a GCP project you can
create tables in, set `REBUILD = True` the first time, then *Runtime → Run all*.
You need read access to `tandem-interview.takehome_wu_timothy`. A full rebuild scans about 4 GB (≈$0.03).

**Layers**

| Layer | Object | Built by |
|---|---|---|
| Source | `tandem-interview.takehome_wu_timothy.*` | read-only |
| Clean | `biologic_ref`, `claims_enriched` | `sql/01_build_analytics_layer.sql` |
| Mart | `npi_summary` | `sql/02_npi_summary.sql` |
| Mart | `specialty_segments`, `sales_targets` (views) | `sql/03_specialty_segments.sql` |
| Economics | pandas, from the assumptions cell | `forus_gtm.py` |

SQL produces **volumes only**; every dollar and minute is applied in Python so scenarios never need SQL changes.

## 1. Configuration

In [ ]:
BILLING_PROJECT = "tim-wu-personal"                  # project that runs and pays for queries
TARGET          = "tim-wu-personal.forus_analytics"  # project.dataset where the analytics layer lives
SOURCE          = "tandem-interview.takehome_wu_timothy"
REBUILD         = False   # True = re-run sql/01-03 (required the first time in a new TARGET)
RUN_DATA_CHECKS = False   # True = run sql/00_data_checks.sql (~1 GB scan)

# Where the repo files live. In Colab, either set REPO_URL to clone, or upload the
# repo folder / mount Drive and point REPO_DIR at it.
REPO_URL = "https://github.com/timothywu11/forus.git"  # set to "" if you uploaded the files instead
REPO_DIR = "."

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    if REPO_URL:
        if not os.path.exists("forus-takehome"):
            subprocess.run(["git", "clone", "-q", REPO_URL, "forus-takehome"], check=True)
        REPO_DIR = "forus-takehome"

sys.path.insert(0, REPO_DIR)
SQL_DIR = os.path.join(REPO_DIR, "sql")
assert os.path.exists(os.path.join(SQL_DIR, "01_build_analytics_layer.sql")), \
    f"SQL files not found under {SQL_DIR}; set REPO_URL or REPO_DIR"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import forus_gtm as fg

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 40)
client = bigquery.Client(project=BILLING_PROJECT)

## 2. Build the analytics layer

In [ ]:
def bq(sql: str) -> pd.DataFrame:
    # Run one query and return a DataFrame.
    return client.query(sql).to_dataframe()

def run_sql_file(name: str) -> pd.DataFrame:
    # Run a multi-statement script from sql/, with {target} filled in.
    # Returns the result of the script's last SELECT (its validation query).
    with open(os.path.join(SQL_DIR, name)) as f:
        script = f.read().replace("{target}", TARGET)
    job = client.query(script)
    result = job.result()
    gb = (job.total_bytes_processed or 0) / 1e9
    print(f"{name}: done, {gb:,.2f} GB processed")
    return result.to_dataframe()

def table_exists(name: str) -> bool:
    try:
        client.get_table(f"{TARGET}.{name}")
        return True
    except NotFound:
        return False

if RUN_DATA_CHECKS:
    display(run_sql_file("00_data_checks.sql"))

if REBUILD or not table_exists("npi_summary"):
    display(run_sql_file("01_build_analytics_layer.sql"))
    display(run_sql_file("02_npi_summary.sql"))
    run_sql_file("03_specialty_segments.sql")
else:
    print(f"Reusing existing tables in {TARGET} (set REBUILD = True to rebuild)")

## 3. Validate

Every layer must reconcile to the source: same patient-drug pairs, no rows lost or duplicated.
A mismatch here means a join fanned out or dropped rows; stop and investigate before reading any result below.

In [ ]:
checks = bq(f'''
SELECT
  (SELECT COUNT(*)               FROM `{SOURCE}.claims_fact`)          AS source_rows,
  (SELECT COUNT(*)               FROM `{TARGET}.claims_enriched`)      AS enriched_rows,
  (SELECT SUM(npi_drug_patients) FROM `{SOURCE}.claims_fact`)          AS source_pairs,
  (SELECT SUM(total_pairs)       FROM `{TARGET}.npi_summary`)          AS npi_summary_pairs,
  (SELECT SUM(biologic_pairs + other_pairs) FROM `{TARGET}.specialty_segments`) AS segment_pairs,
  (SELECT COUNT(DISTINCT hashed_npi) FROM `{SOURCE}.claims_fact`)      AS source_npis,
  (SELECT COUNT(*)               FROM `{TARGET}.npi_summary`)          AS npi_summary_rows
''').T.rename(columns={0: "value"})
display(checks)

v = checks["value"]
assert v["source_rows"] == v["enriched_rows"], "claims_enriched row count != claims_fact"
assert v["source_pairs"] == v["npi_summary_pairs"] == v["segment_pairs"], "patient-drug pairs do not reconcile"
assert v["source_npis"] == v["npi_summary_rows"], "npi_summary is not one row per NPI"
print("All reconciliation checks passed.")

## 4. Load the marts

In [ ]:
segments = bq(f"SELECT * FROM `{TARGET}.specialty_segments`")
targets  = bq(f"SELECT * FROM `{TARGET}.sales_targets` WHERE is_candidate_specialty")
print(f"segments: {len(segments):,} rows | sales targets (candidate specialties): {len(targets):,} rows")

segments.groupby("specialty")[["n_providers", "n_biologic_prescribers", "n_sales_targets",
                               "biologic_pairs", "other_pairs"]].sum()

## 5. Assumptions

**Change numbers here, then re-run the cells below.** Prices and servicing minutes come from the prompt;
everything else is an assumption to state in the write-up and pressure-test in §8.

- `scripts_per_pair_*` converts claims patient-drug pairs into serviced scripts per year. If both tiers use
  the same value it rescales every specialty equally and cannot change the ranking.
- `capture_rate` is the share of an onboarded provider's scripts that flow through Forus.
- `cost_per_sales_target` is one sales cycle per institution or independent practice;
  `cost_per_provider_onboard` is per-provider setup and training.
- `biologic_definition`: `"base"` = primer list + biosimilars; `"sensitivity"` adds Rinvoq, Kevzara, etc.

In [ ]:
A = {
    **fg.DEFAULT_ASSUMPTIONS,
    # override anything here, e.g.
    # "ops_hourly_cost": 55,
    # "capture_rate": 0.4,
}
pd.Series(A, name="value").to_frame()

## 6. Unit economics

Per-script margin by tier. The break-even column is the fully loaded Ops cost per hour at which a tier stops
covering its own servicing time.

In [ ]:
fg.unit_economics(A)

## 7. Specialty scorecard

One row per metric, assuming Forus onboards every provider in the specialty. Internal Medicine is shown as a
low-biologic baseline, not a candidate. How to read the rows:

- **Size:** `providers`, `revenue`, `contribution` (revenue − servicing cost, annual).
- **Value mix:** `biologic_share_of_revenue` (value to pharma partners and to Forus).
- **Acquisition efficiency:** `contribution_per_sales_target`, `providers_per_sales_target`, `payback_months`.
- **Ops load:** `ops_fte`, `contribution_per_ops_hour`.

In [ ]:
scorecard = fg.specialty_scorecard(segments, A)
scorecard.T

## 8. Capture curve: what does the first N sales targets buy?

Forus will not onboard a whole specialty at once. Rank each specialty's sales targets (institutions or
independent practices) by year-1 ROI, then accumulate. The steeper curve is the specialty where a limited
GTM team builds a profitable footprint fastest.

In [ ]:
curve = fg.capture_curve(targets, A)

COLORS = {"Dermatology": "#2a78d6", "Gastroenterology": "#eb6834", "Rheumatology": "#1baf7a"}
X_MAX = 1500

fig, ax = plt.subplots(figsize=(9, 5))
for spec, c in curve.groupby("specialty"):
    c = c[c["cum_n_sales_targets"] <= X_MAX]
    ax.plot(c["cum_n_sales_targets"], c["cum_contribution"] / 1e6,
            color=COLORS[spec], linewidth=2, label=spec)
    ax.annotate(spec, (c["cum_n_sales_targets"].iloc[-1], c["cum_contribution"].iloc[-1] / 1e6),
                xytext=(6, 0), textcoords="offset points", va="center", fontsize=10, color="#333333")
ax.set_xlabel("Sales targets closed (best year-1 ROI first)")
ax.set_ylabel("Cumulative annual contribution ($M)")
ax.set_title("Annual contribution unlocked per sales target closed", loc="left", fontsize=12)
ax.grid(axis="y", color="#e6e6e6", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#bbbbbb")
ax.legend(frameon=False, loc="upper left")
ax.set_xlim(0, X_MAX * 1.18)
plt.tight_layout()
plt.show()

fg.at_budget(curve, [50, 100, 250, 500])

## 9. Sensitivity: does the recommendation survive the assumptions?

Each table varies one assumption and shows which specialty leads. If the leader never changes across a
plausible range, the recommendation is robust to that assumption.

In [ ]:
sens = {
    "ops_hourly_cost (contribution per sales target)":
        fg.sensitivity(segments, "ops_hourly_cost", [25, 40, 60, 80], A),
    "minutes_biologic (contribution per sales target)":
        fg.sensitivity(segments, "minutes_biologic", [30, 60, 90], A),
    "price_biologic (contribution per sales target)":
        fg.sensitivity(segments, "price_biologic", [100, 200, 300], A),
    "service_other_scripts (contribution per sales target)":
        fg.sensitivity(segments, "service_other_scripts", [True, False], A),
    "biologic_definition (contribution per sales target)":
        fg.sensitivity(segments, "biologic_definition", ["base", "sensitivity"], A),
    "cost_per_sales_target (payback months)":
        fg.sensitivity(segments, "cost_per_sales_target", [1000, 5000, 20000], A, metric="payback_months"),
}
for name, table in sens.items():
    print(name)
    display(table)

## 10. Targeting starter (Part 1b)

Set `FOCUS` to the recommended specialty. The first table shows where value concentrates by institution size
and type; the second is the ranked target list the GTM team would work from.

In [ ]:
FOCUS = "Rheumatology"

by_size = (fg.add_economics(segments[segments["specialty"] == FOCUS], A)
             .groupby("institution_size")
             .agg(providers=("n_providers", "sum"), sales_targets=("n_sales_targets", "sum"),
                  biologic_pairs=("biologic_pairs", "sum"), contribution=("contribution", "sum"),
                  acquisition_cost=("acquisition_cost", "sum")))
by_size["contribution_per_target"] = by_size["contribution"] / by_size["sales_targets"]
by_size["share_of_contribution"] = by_size["contribution"] / by_size["contribution"].sum()
by_size["roi_year1"] = by_size["contribution"] / by_size["acquisition_cost"]
display(by_size)

fg.rank_targets(targets, FOCUS, A, top=25)

## Notes and caveats

- **Units.** Claims volumes are patient-drug pairs, not scripts and not unique patients.
- **Coverage.** 7,919 NPIs (≈5% of patient-drug pairs) have no NPPES specialty and are excluded from the
  specialty comparison; 12.6% of claims rows carry no institution. Providers with no institution on any claim are treated as
  independent practices (one sales target each); other providers' unaffiliated claims roll up to their primary
  institution.
- **Institution size** uses `affiliation_npi_count`, which counts every provider at the institution, not only
  these specialties.
- **Rheum/GI undercount.** Some rheumatologists and gastroenterologists register in NPPES as Internal Medicine,
  so those specialties' totals are conservative.
- Full list of source-data issues and how each is handled: header of `sql/01_build_analytics_layer.sql`.